In [ ]:
import pandas as pd

activities_clean = pd.read_csv(
    "../data/processed/activities_clean.csv"
)

compounds = pd.read_csv(
    "../data/raw/chembl_leishmania_compounds.csv",
    sep=";",
    engine="python",
    on_bad_lines="skip"
)
compounds.columns.tolist()

['Compound ChEMBL ID',
 'Name',
 'Synonyms',
 'Type',
 'Max Phase',
 'Molecular Weight',
 'Targets',
 'Bioactivities',
 'AlogP',
 'Polar Surface Area',
 'HBA',
 'HBD',
 '#RO5 Violations',
 '#Rotatable Bonds',
 'Passes Ro3',
 'QED Weighted',
 'Aromatic Rings',
 'Structure Type',
 'Inorganic Flag',
 'Heavy Atoms',
 'Np Likeness Score',
 'Molecular Formula',
 'Smiles',
 'Inchi Key',
 'Inchi',
 'Withdrawn Flag',
 'Orphan',
 'Records Key',
 'Records Name']

## Merging Activity and Descriptor Data

The cleaned activity dataset was merged with the compound descriptor dataset using ChEMBL compound identifiers. This step combined biological activity information (pIC50) with molecular descriptors required for machine learning model development.

In [9]:
merged_df = pd.merge(
    activities_clean,
    compounds,
    left_on="Molecule ChEMBL ID",
    right_on="Compound ChEMBL ID",
    how="inner"
)
print(merged_df.shape)
merged_df.head()

(5533, 32)


,Molecule ChEMBL ID,pIC50,Smiles_x,Compound ChEMBL ID,Name,Synonyms,Type,Max Phase,Molecular Weight,Targets,...,Heavy Atoms,Np Likeness Score,Molecular Formula,Smiles_y,Inchi Key,Inchi,Withdrawn Flag,Orphan,Records Key,Records Name
0,CHEMBL1000,3.940555,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,CHEMBL1000,CETIRIZINE,AC-170|CETIDERM|CETIRIZINA|CETIRIZINE,Small molecule,4.0,388.90,121,...,27.0,-1.21,C21H25ClN2O3,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,ZKLPARSLTMPFCP-UHFFFAOYSA-N,InChI=1S/C21H25ClN2O3/c22-19-8-6-18(7-9-19)21(...,False,0,"['6', 'Cetirizine', 'Cetirizine', 'Cetirizine'...","['Cetirizine', '(2-{4-[(4-Chloro-phenyl)-pheny..."
1,CHEMBL100210,4.443697,CCCC[C@H]1C(=O)O[C@@H]2O[C@@]3(CC)CC[C@H]4[C@H...,CHEMBL100210,NaN,NaN,Small molecule,NaN,338.44,2,...,24.0,2.52,C19H30O5,CCCC[C@H]1C(=O)O[C@@H]2O[C@@]3(CC)CC[C@H]4[C@H...,SEKJKOCIXZWYFI-MDBUGZKMSA-N,InChI=1S/C19H30O5/c1-4-6-7-13-15-9-8-12(3)14-1...,False,-1,"['31', 'SI, artemisin38']","['Octahydro-3-ethyl-6-methyl-9-butyl-3,12-epox..."
2,CHEMBL100740,3.800519,C[C@@H]1CC[C@H]2[C@@H](C)C(=O)O[C@@H]3OC(C)(C)...,CHEMBL100740,NaN,NaN,Small molecule,NaN,284.35,3,...,20.0,2.66,C15H24O5,C[C@@H]1CC[C@H]2[C@@H](C)C(=O)O[C@@H]3OC(C)(C)...,FFRVSIQAOWTEAA-OVCRQHFTSA-N,InChI=1S/C15H24O5/c1-8-6-7-11-9(2)12(16)17-13-...,False,-1,"['72', '5j', '53']","['(1R,6S,9R,10S,13R,14S)-4,4,9,13,14-Pentameth..."
3,CHEMBL104,5.242633,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,CHEMBL104,CLOTRIMAZOLE,ABTRIM|ACTAVALL|BAY 5097|BAY-5097|CANDIDEN|CAN...,Small molecule,4.0,344.85,437,...,25.0,-0.58,C22H17ClN2,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,VNFPBHJOKIVQEB-UHFFFAOYSA-N,InChI=1S/C22H17ClN2/c23-21-14-8-7-13-20(21)22(...,False,0,"['clotrimazole', '1, CLT, clotrimazole', 'Clot...",['1-((2-chlorophenyl)diphenylmethyl)-1H-imidaz...
4,CHEMBL106,3.898138,OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F,CHEMBL106,FLUCONAZOLE,ALKANAZOLE|BAYT-006267|BAYT006267|CANESTEN ORA...,Small molecule,4.0,306.28,425,...,22.0,-0.86,C13H12F2N6O,OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F,RFHAOTPXVQNOHP-UHFFFAOYSA-N,InChI=1S/C13H12F2N6O/c14-10-1-2-11(12(15)3-10)...,False,0,"['fluconazole', 'FLZ', 'Fluconazole', 'Flucona...","['2-(2,4-difluorophenyl)-1,3-di(1H-1,2,4-triaz..."


In [10]:
(
    merged_df["Smiles_x"] ==
    merged_df["Smiles_y"]
).value_counts()

True     5525
False       8
Name: count, dtype: int64

## Verification of SMILES Consistency

SMILES representations from the activity and descriptor datasets were compared following the merge. The vast majority of compounds (5,525 out of 5,533) showed identical molecular structures across both datasets, indicating successful matching of compound records. A small number of discrepancies were retained for further inspection.

In [11]:
merged_df[
    merged_df["Smiles_x"] != merged_df["Smiles_y"]
][
    [
        "Molecule ChEMBL ID",
        "Smiles_x",
        "Smiles_y"
    ]
]

,Molecule ChEMBL ID,Smiles_x,Smiles_y
247,CHEMBL1366,NaN,NaN
1103,CHEMBL239129,NaN,NaN
1794,CHEMBL3339150,NaN,NaN
3116,CHEMBL3736283,NaN,NaN
3140,CHEMBL3764926,NaN,NaN
3371,CHEMBL4071680,NaN,NaN
3377,CHEMBL4075089,NaN,NaN
4151,CHEMBL4745267,NaN,NaN


In [12]:
merged_df=merged_df.dropna(subset=['Smiles_x'])
print(merged_df.shape)

(5525, 32)


## Verification of Molecular Structures

SMILES representations from both datasets were compared after merging. All available structures were found to be consistent between datasets. A small number of compounds contained missing SMILES information and were removed prior to model development.

In [13]:
merged_df.isnull().sum().sort_values(
    ascending=False
)

Max Phase             5419
Synonyms              5414
Name                  5252
Type                   470
Polar Surface Area      68
HBA                     68
HBD                     68
AlogP                   68
#RO5 Violations         68
QED Weighted            68
Heavy Atoms             68
Np Likeness Score       68
Aromatic Rings          68
#Rotatable Bonds        68
Passes Ro3              68
pIC50                    0
Targets                  0
Molecular Weight         0
Bioactivities            0
Molecule ChEMBL ID       0
Compound ChEMBL ID       0
Smiles_x                 0
Inorganic Flag           0
Structure Type           0
Molecular Formula        0
Smiles_y                 0
Inchi Key                0
Inchi                    0
Withdrawn Flag           0
Orphan                   0
Records Key              0
Records Name             0
dtype: int64

In [14]:
descriptor_cols = [
    "Molecular Weight",
    "AlogP",
    "Polar Surface Area",
    "HBA",
    "HBD",
    "#RO5 Violations",
    "#Rotatable Bonds",
    "QED Weighted",
    "Aromatic Rings",
    "Heavy Atoms",
    "Np Likeness Score"
]
merged_df = merged_df.dropna(
    subset=descriptor_cols
)

print(merged_df.shape)

(5457, 32)


## Handling Missing Descriptor Values

Several molecular descriptor columns contained a small number of missing values. Because these missing records represented a very small proportion of the dataset, compounds lacking descriptor information were removed to ensure a complete feature set for machine learning analysis.

In [16]:
model_df = merged_df[
    [
        "Molecule ChEMBL ID",
        "Smiles_x",
        "pIC50",
        "Molecular Weight",
        "AlogP",
        "Polar Surface Area",
        "HBA",
        "HBD",
        "#RO5 Violations",
        "#Rotatable Bonds",
        "QED Weighted",
        "Aromatic Rings",
        "Heavy Atoms",
        "Np Likeness Score"
    ]
].copy()

## Selection of Modeling Features

Relevant molecular descriptors and the target activity variable (pIC50) were selected to construct the final modeling dataset. Metadata and non-informative identifiers were excluded, retaining only descriptors expected to contribute to activity prediction.

In [17]:
model_df.rename(
    columns={
        "Smiles_x": "SMILES",
        "Molecular Weight": "Molecular_Weight",
        "Polar Surface Area": "TPSA",
        "#RO5 Violations": "RO5_Violations",
        "#Rotatable Bonds": "Rotatable_Bonds",
        "QED Weighted": "QED",
        "Aromatic Rings": "Aromatic_Rings",
        "Heavy Atoms": "Heavy_Atoms",
        "Np Likeness Score": "NP_Likeness"
    },
    inplace=True
)

model_df.head()

,Molecule ChEMBL ID,SMILES,pIC50,Molecular_Weight,AlogP,TPSA,HBA,HBD,RO5_Violations,Rotatable_Bonds,QED,Aromatic_Rings,Heavy_Atoms,NP_Likeness
0,CHEMBL1000,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,3.940555,388.90,3.15,53.01,4.0,1.0,0.0,8.0,0.70,2.0,27.0,-1.21
1,CHEMBL100210,CCCC[C@H]1C(=O)O[C@@H]2O[C@@]3(CC)CC[C@H]4[C@H...,4.443697,338.44,3.96,53.99,5.0,0.0,0.0,4.0,0.57,0.0,24.0,2.52
2,CHEMBL100740,C[C@@H]1CC[C@H]2[C@@H](C)C(=O)O[C@@H]3OC(C)(C)...,3.800519,284.35,2.64,53.99,5.0,0.0,0.0,0.0,0.51,0.0,20.0,2.66
3,CHEMBL104,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,5.242633,344.85,5.38,17.82,2.0,0.0,1.0,4.0,0.45,4.0,25.0,-0.58
4,CHEMBL106,OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F,3.898138,306.28,0.74,81.65,7.0,1.0,0.0,5.0,0.75,3.0,22.0,-0.86


## Standardization of Feature Names

Descriptor column names were standardized to improve readability and facilitate downstream analysis and model development.

In [ ]:
model_df.to_csv(
    "../data/processed/leishmania_ml_dataset.csv",
    index=False
)

In [19]:
model_df.head()

,Molecule ChEMBL ID,SMILES,pIC50,Molecular_Weight,AlogP,TPSA,HBA,HBD,RO5_Violations,Rotatable_Bonds,QED,Aromatic_Rings,Heavy_Atoms,NP_Likeness
0,CHEMBL1000,O=C(O)COCCN1CCN(C(c2ccccc2)c2ccc(Cl)cc2)CC1,3.940555,388.90,3.15,53.01,4.0,1.0,0.0,8.0,0.70,2.0,27.0,-1.21
1,CHEMBL100210,CCCC[C@H]1C(=O)O[C@@H]2O[C@@]3(CC)CC[C@H]4[C@H...,4.443697,338.44,3.96,53.99,5.0,0.0,0.0,4.0,0.57,0.0,24.0,2.52
2,CHEMBL100740,C[C@@H]1CC[C@H]2[C@@H](C)C(=O)O[C@@H]3OC(C)(C)...,3.800519,284.35,2.64,53.99,5.0,0.0,0.0,0.0,0.51,0.0,20.0,2.66
3,CHEMBL104,Clc1ccccc1C(c1ccccc1)(c1ccccc1)n1ccnc1,5.242633,344.85,5.38,17.82,2.0,0.0,1.0,4.0,0.45,4.0,25.0,-0.58
4,CHEMBL106,OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F,3.898138,306.28,0.74,81.65,7.0,1.0,0.0,5.0,0.75,3.0,22.0,-0.86
